# AkaVision: classificador do tipo de documento

Rede neural simples que lê o começo de um documento e diz de qual dos 4 tipos ele é:

| Tipo | O que é |
|---|---|
| **Técnica** | explica como algo funciona ou como fazer: manual, handbook, guia, procedimento |
| **Jurídica** | ato ou decisão com força de lei: lei, decreto, regra publicada, contrato, decisão judicial |
| **Normativa** | norma ou especificação que define requisitos: MIL-STD, FED-STD, ABNT, especificação técnica |
| **Quantitativa** | o conteúdo principal são dados: tabelas, resultados de ensaio, estatísticas |

**Caminho dos dados**

1. `classificador/coletar_amostras.py` (no repositório do back) baixa PDFs públicos (EverySpec, NASA NTRS, GovInfo, Federal Register) e gera `amostras.csv` com o texto de cada um.
2. A equipe confere cada linha e preenche a coluna `tipo`.
3. **Este notebook**: texto → vetor de 1024 números (bge-m3, o mesmo modelo de embedding do RAG) → rede neural 1024 → 64 → 4 → probabilidade de cada tipo.
4. Os pesos treinados saem num arquivo `modelo_tipo_documento.npz`, que vai para `classificador/` no back. No upload, o back faz a mesma conta com numpy e o formulário já vem com o tipo sugerido.

> Ambiente: Colab com GPU (Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU).

## 1. Bibliotecas

In [ ]:
!pip -q install sentence-transformers

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sentence_transformers import SentenceTransformer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Mesma ordem que vai para o .npz e para o back.
TIPOS = ["Técnica", "Jurídica", "Normativa", "Quantitativa"]
SEMENTE = 42
np.random.seed(SEMENTE)
tf.random.set_seed(SEMENTE)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Carregar a planilha rotulada

Envie o `amostras.csv` com a coluna `tipo` preenchida (baixe do Google Planilhas como CSV).
Só entram as linhas cujo `tipo` é um dos 4; "descartar" e vazio ficam de fora.

In [ ]:
from google.colab import files

enviado = files.upload()  # escolha o amostras.csv
arquivo_csv = next(iter(enviado))
dados = pd.read_csv(arquivo_csv, encoding="utf-8-sig")
print("Linhas no CSV:", len(dados))

dados["tipo"] = dados["tipo"].fillna("").str.strip()
rotulados = dados[dados["tipo"].isin(TIPOS)].reset_index(drop=True)
print("Rotulados (entram no treino):", len(rotulados))
print("Sem rótulo ou descartados:", len(dados) - len(rotulados))
rotulados.head()

### Quantos documentos de cada tipo

In [ ]:
contagem = rotulados["tipo"].value_counts().reindex(TIPOS, fill_value=0)
print(contagem)

# Cada tipo precisa de exemplos para treinar e para testar.
MINIMO_POR_TIPO = 10
poucos = contagem[contagem < MINIMO_POR_TIPO]
if len(poucos):
    raise ValueError(
        f"Faltam documentos rotulados (mínimo {MINIMO_POR_TIPO} por tipo): "
        + ", ".join(f"{tipo} tem {total}" for tipo, total in poucos.items())
        + ". Complete a coluna 'tipo' (com acentos) e envie o CSV de novo."
    )

contagem.plot(kind="bar", color="#4C78A8", figsize=(6, 3), rot=0)
plt.title("Documentos rotulados por tipo")
plt.ylabel("documentos")
plt.show()

# Quanto o rótulo da equipe concordou com o palpite pela fonte.
pd.crosstab(rotulados["tipo_sugerido"], rotulados["tipo"])

## 3. Texto → vetor (embedding bge-m3)

Cada documento vira um vetor de 1024 números normalizado (comprimento 1).
O back gera o mesmo vetor com o bge-m3 do Ollama, na VM da IA. Os pesos são os mesmos (BAAI/bge-m3), então a rede treinada aqui serve lá.
O texto já vem cortado no CSV (4000 caracteres do começo), pela mesma função que o back usa no upload.

In [ ]:
modelo_embedding = SentenceTransformer(
    "BAAI/bge-m3", device="cuda" if tf.config.list_physical_devices("GPU") else "cpu"
)
X = modelo_embedding.encode(
    rotulados["texto"].tolist(),
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)
y = rotulados["tipo"].map(TIPOS.index).to_numpy()
print("X:", X.shape, "| y:", y.shape)

## 4. Treino e teste

80% dos documentos treinam a rede e 20% ficam guardados para medir o resultado no fim (estratificado: a mesma proporção de cada tipo nos dois lados).

In [ ]:
X_treino, X_teste, y_treino, y_teste, idx_treino, idx_teste = train_test_split(
    X, y, np.arange(len(y)), test_size=0.2, stratify=y, random_state=SEMENTE
)
print("Treino:", len(y_treino), "| Teste:", len(y_teste))

# Tipo com menos exemplos pesa mais no erro, para a rede não ignorá-lo.
pesos = compute_class_weight("balanced", classes=np.arange(len(TIPOS)), y=y_treino)
peso_por_classe = dict(enumerate(pesos))
peso_por_classe

## 5. A rede neural

- **Entrada**: os 1024 números do embedding
- **Camada oculta**: 64 neurônios com ReLU
- **Dropout 30%**: durante o treino desliga neurônios ao acaso, para a rede não decorar os exemplos
- **Saída**: 4 neurônios com softmax, a probabilidade de cada tipo (somam 1)

O treino para sozinho quando o erro na validação para de cair (EarlyStopping) e volta para os melhores pesos.

In [ ]:
def criar_rede():
    return tf.keras.Sequential(
        [
            tf.keras.Input(shape=(X.shape[1],)),
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(len(TIPOS), activation="softmax"),
        ]
    )


rede = criar_rede()
rede.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
rede.summary()

parada = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=15, restore_best_weights=True
)
historico = rede.fit(
    X_treino,
    y_treino,
    validation_split=0.2,
    epochs=300,
    batch_size=16,
    class_weight=peso_por_classe,
    callbacks=[parada],
    verbose=0,
)
melhor_epoca = int(np.argmin(historico.history["val_loss"])) + 1
print(
    "Épocas rodadas:", len(historico.history["loss"]), "| melhor época:", melhor_epoca
)

### Curvas de aprendizado

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
a1.plot(historico.history["loss"], label="treino")
a1.plot(historico.history["val_loss"], label="validação")
a1.set_title("Erro (loss)")
a1.set_xlabel("época")
a1.legend()
a2.plot(historico.history["accuracy"], label="treino")
a2.plot(historico.history["val_accuracy"], label="validação")
a2.set_title("Acurácia")
a2.set_xlabel("época")
a2.legend()
plt.show()

## 6. Resultado nos 20% de teste

Documentos que a rede nunca viu.
- **Precisão**: dos que a rede disse ser do tipo X, quantos eram X.
- **Recall**: dos que eram X, quantos a rede acertou.
- **Matriz de confusão**: linha = tipo real, coluna = tipo previsto. A diagonal são os acertos.

In [ ]:
probabilidades = rede.predict(X_teste, verbose=0)
previsto = probabilidades.argmax(axis=1)

print(f"Acurácia no teste: {accuracy_score(y_teste, previsto):.1%}\n")
print(
    classification_report(
        y_teste, previsto, labels=range(len(TIPOS)), target_names=TIPOS, zero_division=0
    )
)

ConfusionMatrixDisplay.from_predictions(
    y_teste,
    previsto,
    labels=range(len(TIPOS)),
    display_labels=TIPOS,
    cmap="Blues",
    colorbar=False,
)
plt.title("Matriz de confusão (teste)")
plt.show()

### Onde a rede errou

In [ ]:
erros = rotulados.iloc[idx_teste].assign(
    previsto=[TIPOS[i] for i in previsto],
    confianca=probabilidades.max(axis=1).round(2),
)
erros = erros[erros["tipo"] != erros["previsto"]]
print(len(erros), "erro(s) no teste")
erros[["titulo", "fonte", "tipo", "previsto", "confianca"]]

## 7. Modelo final e exportação

Com o resultado medido, a rede é treinada de novo com **todos** os documentos rotulados, pelo mesmo número de épocas da melhor rodada.
Os pesos vão para `modelo_tipo_documento.npz`:

| chave | o que é |
|---|---|
| `W0`, `b0` | pesos e viés da camada oculta (1024 × 64) |
| `W1`, `b1` | pesos e viés da saída (64 × 4) |
| `classes` | os 4 tipos, na ordem da saída |
| `modelo_embedding` | `bge-m3` (o modelo que o back pede ao Ollama) |

O Dropout só age no treino: no uso ele não muda nada, por isso não vai para o arquivo.

In [ ]:
final = criar_rede()
final.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
pesos_todos = compute_class_weight("balanced", classes=np.arange(len(TIPOS)), y=y)
final.fit(
    X,
    y,
    epochs=melhor_epoca,
    batch_size=16,
    class_weight=dict(enumerate(pesos_todos)),
    verbose=0,
)

densas = [
    camada for camada in final.layers if isinstance(camada, tf.keras.layers.Dense)
]
camadas = {}
for i, camada in enumerate(densas):
    W, b = camada.get_weights()
    camadas[f"W{i}"], camadas[f"b{i}"] = W.astype(np.float32), b.astype(np.float32)

np.savez(
    "modelo_tipo_documento.npz",
    classes=np.array(TIPOS),
    modelo_embedding="bge-m3",
    **camadas,
)


# Conferência: a conta em numpy (a mesma do back) dá o mesmo que o Keras.
def prever_numpy(x):
    for i in range(len(densas)):
        x = x @ camadas[f"W{i}"] + camadas[f"b{i}"]
        if i < len(densas) - 1:
            x = np.maximum(x, 0)
    x = np.exp(x - x.max(axis=1, keepdims=True))
    return x / x.sum(axis=1, keepdims=True)


diferenca = np.abs(prever_numpy(X[:20]) - final.predict(X[:20], verbose=0)).max()
print("Maior diferença numpy x Keras:", diferenca)
assert diferenca < 1e-4

files.download("modelo_tipo_documento.npz")

## 8. Colocar no sistema

1. Copie o `modelo_tipo_documento.npz` para a pasta `classificador/` do repositório do back e abra um PR.
2. Depois do merge, o deploy leva o arquivo para a VM do back. A próxima sugestão já usa a rede nova, sem reiniciar.
3. No upload, ao escolher o PDF, o formulário chama `POST /api/documents/sugerir-tipo/` e pré-preenche o tipo quando a confiança é de 50% ou mais. O usuário confirma ou troca.

Para retreinar (mais documentos rotulados, tipo novo): rode o coletor de novo, rotule as linhas novas e rode este notebook outra vez.